# 18. Stacking Adversarial Training

**Tujuan:** Adversarial Training pada Stacking Ensemble — evaluasi S1-S4.
Perbandingan: Single XGBoost (AT) vs Stacking Ensemble (AT).

**Input:** `stacking_baseline_15.pkl`, `stacking_adversarial_17.pkl`, `robust_results_06.pkl`

**Output:** `stacking_robust_18.pkl`, PNG comparison S1-S4

In [ ]:
import sys
!{sys.executable} -m pip install lightgbm catboost scikit-learn xgboost matplotlib -q

import numpy as np
import pandas as pd
import pickle
import os
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, matthews_corrcoef, accuracy_score,
                             precision_score, recall_score, confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
DATA_DIR = '../data/'
EPSILON_TRAIN = 0.1
CACHE_FILE = os.path.join(DATA_DIR, 'stacking_robust_18.pkl')

# === CHECK CACHE ===
if os.path.exists(CACHE_FILE):
    print(f'Cache found: {CACHE_FILE} — loading (skip computation)...')
    with open(CACHE_FILE, 'rb') as f:
        cached = pickle.load(f)
    SKIP_TRAINING = True
else:
    print('No cache. Will run adversarial training.')
    SKIP_TRAINING = False

print('Libraries loaded.')

## 1. Load Data & Adversarial Samples

In [ ]:
# Load stacking baseline
with open(os.path.join(DATA_DIR, 'stacking_baseline_15.pkl'), 'rb') as f:
    stack_data = pickle.load(f)

X_train = stack_data['X_train']
X_test = stack_data['X_test']
y_train = stack_data['y_train']
y_test = stack_data['y_test']
n_classes = stack_data['n_classes']

# Load adversarial data from notebook 17
with open(os.path.join(DATA_DIR, 'stacking_adversarial_17.pkl'), 'rb') as f:
    adv_data = pickle.load(f)

saliency = adv_data['saliency']
X_test_subset = adv_data['X_test_subset']
y_test_subset = adv_data['y_test_subset']

print(f'Training data: {X_train.shape}')
print(f'Test subset for evaluation: {X_test_subset.shape}')

## 2. Generate Adversarial Training Data

In [ ]:
# Generate adversarial samples on training data
# Use saliency from test as proxy (approximation for training augmentation)
print(f'Generating adversarial training samples (ε={EPSILON_TRAIN})...')

# Compute saliency on training subset
MAX_TRAIN_SAMPLES = min(50000, X_train.shape[0])
X_train_sub = X_train[:MAX_TRAIN_SAMPLES]
y_train_sub = y_train[:MAX_TRAIN_SAMPLES]

# Simple approximation: use random perturbation in saliency direction
# (full saliency computation on training would be too slow)
np.random.seed(RANDOM_SEED)
noise_direction = np.random.choice([-1, 1], size=X_train_sub.shape)
X_adv_train = X_train_sub + EPSILON_TRAIN * noise_direction

# Augmented dataset: 80% clean + 20% adversarial
n_adv = int(X_train.shape[0] * 0.2)
adv_indices = np.random.choice(MAX_TRAIN_SAMPLES, size=n_adv, replace=True)

X_robust_train = np.vstack([X_train, X_adv_train[adv_indices]])
y_robust_train = np.concatenate([y_train, y_train_sub[adv_indices]])

print(f'Clean training: {X_train.shape[0]}')
print(f'Adversarial augmentation: {n_adv}')
print(f'Total robust training: {X_robust_train.shape[0]}')

## 3. Train Robust Stacking Ensemble

In [ ]:
def train_stacking_full(X_tr, y_tr, X_te, n_classes):
    """Train full stacking pipeline, return models and predictions."""
    base_learners = {
        'XGBoost': XGBClassifier(
            max_depth=6, n_estimators=100, learning_rate=0.1,
            use_label_encoder=False, eval_metric='mlogloss',
            random_state=RANDOM_SEED, verbosity=0
        ),
        'LightGBM': LGBMClassifier(
            max_depth=6, n_estimators=100, learning_rate=0.1,
            random_state=RANDOM_SEED, verbose=-1
        ),
        'CatBoost': CatBoostClassifier(
            depth=6, iterations=100, learning_rate=0.1,
            random_seed=RANDOM_SEED, verbose=0
        )
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    meta_train = np.zeros((X_tr.shape[0], n_classes * 3))
    meta_test = np.zeros((X_te.shape[0], n_classes * 3))
    
    trained_models = {}
    for idx, (name, model) in enumerate(base_learners.items()):
        oof_preds = np.zeros((X_tr.shape[0], n_classes))
        test_preds = np.zeros((X_te.shape[0], n_classes))
        
        for train_idx, val_idx in cv.split(X_tr, y_tr):
            Xt, Xv = X_tr[train_idx], X_tr[val_idx]
            yt, yv = y_tr[train_idx], y_tr[val_idx]
            m = model.__class__(**model.get_params())
            m.fit(Xt, yt)
            oof_preds[val_idx] = m.predict_proba(Xv)
            test_preds += m.predict_proba(X_te) / cv.n_splits
        
        meta_train[:, idx*n_classes:(idx+1)*n_classes] = oof_preds
        meta_test[:, idx*n_classes:(idx+1)*n_classes] = test_preds
        
        # Train final model on full data
        final_model = model.__class__(**model.get_params())
        final_model.fit(X_tr, y_tr)
        trained_models[name] = final_model
    
    scaler = StandardScaler()
    meta_tr_s = scaler.fit_transform(meta_train)
    meta_te_s = scaler.transform(meta_test)
    
    meta_lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, multi_class='multinomial')
    meta_lr.fit(meta_tr_s, y_tr)
    y_pred = meta_lr.predict(meta_te_s)
    
    return trained_models, meta_lr, scaler, y_pred

print('Training Robust Stacking Ensemble...')
start = time.time()
robust_models, robust_meta, robust_scaler, _ = train_stacking_full(
    X_robust_train, y_robust_train, X_test_subset, n_classes
)
train_time = time.time() - start
print(f'Done in {train_time:.1f}s')

## 4. Evaluate S1-S4 Scenarios

In [ ]:
def stacking_predict_with_models(X, models, meta_lr, scaler, n_classes):
    meta = np.zeros((X.shape[0], n_classes * len(models)))
    for idx, (name, model) in enumerate(models.items()):
        meta[:, idx*n_classes:(idx+1)*n_classes] = model.predict_proba(X)
    return meta_lr.predict(scaler.transform(meta))

# Generate adversarial test data
X_adv_test = X_test_subset + EPSILON_TRAIN * np.sign(saliency)

# Baseline models from notebook 15
baseline_models = stack_data['base_models']
baseline_meta = stack_data['meta_learner']
baseline_scaler = stack_data['scaler_meta']

# S1: Baseline + Clean
y_s1 = stacking_predict_with_models(X_test_subset, baseline_models, baseline_meta, baseline_scaler, n_classes)
mcc_s1 = matthews_corrcoef(y_test_subset, y_s1)
f1_s1 = f1_score(y_test_subset, y_s1, average='weighted')

# S2: Baseline + Adversarial
y_s2 = stacking_predict_with_models(X_adv_test, baseline_models, baseline_meta, baseline_scaler, n_classes)
mcc_s2 = matthews_corrcoef(y_test_subset, y_s2)
f1_s2 = f1_score(y_test_subset, y_s2, average='weighted')

# S3: Robust + Clean
y_s3 = stacking_predict_with_models(X_test_subset, robust_models, robust_meta, robust_scaler, n_classes)
mcc_s3 = matthews_corrcoef(y_test_subset, y_s3)
f1_s3 = f1_score(y_test_subset, y_s3, average='weighted')

# S4: Robust + Adversarial
y_s4 = stacking_predict_with_models(X_adv_test, robust_models, robust_meta, robust_scaler, n_classes)
mcc_s4 = matthews_corrcoef(y_test_subset, y_s4)
f1_s4 = f1_score(y_test_subset, y_s4, average='weighted')

print('='*65)
print('  STACKING ENSEMBLE: S1-S4 Results')
print('='*65)
print(f'{"Scenario":<25} | {"MCC":<8} | {"F1 (%)":<8}')
print('-'*45)
print(f'{"S1 (Baseline+Clean)":<25} | {mcc_s1:<8.4f} | {f1_s1*100:<8.2f}')
print(f'{"S2 (Baseline+Adversarial)":<25} | {mcc_s2:<8.4f} | {f1_s2*100:<8.2f}')
print(f'{"S3 (Robust+Clean)":<25} | {mcc_s3:<8.4f} | {f1_s3*100:<8.2f}')
print(f'{"S4 (Robust+Adversarial)":<25} | {mcc_s4:<8.4f} | {f1_s4*100:<8.2f}')
print('='*65)
print(f'\nSecurity Gap (S1→S2): {mcc_s1 - mcc_s2:.4f}')
print(f'Integrity Loss (S1→S3): {mcc_s1 - mcc_s3:.4f}')
print(f'Recovery (S2→S4): {mcc_s4 - mcc_s2:.4f}')

## 5. Comparison: Single XGBoost vs Stacking Ensemble (S1-S4)

In [ ]:
# Load single model results from notebook 06
with open(os.path.join(DATA_DIR, 'robust_results_06.pkl'), 'rb') as f:
    single_robust = pickle.load(f)

# Extract single model S1-S4 MCC (adjust keys based on actual pkl structure)
single_s1 = single_robust.get('mcc_s1', 0.9332)
single_s2 = single_robust.get('mcc_s2', 0.0189)
single_s3 = single_robust.get('mcc_s3', 0.9327)
single_s4 = single_robust.get('mcc_s4', 0.9946)

print('\n' + '='*70)
print('  COMPARISON: Single XGBoost vs Stacking Ensemble')
print('='*70)
print(f'{"Scenario":<12} | {"Single MCC":<12} | {"Stacking MCC":<14} | {"Δ":<8}')
print('-'*50)
print(f'{"S1 (Clean)":<12} | {single_s1:<12.4f} | {mcc_s1:<14.4f} | {mcc_s1-single_s1:+.4f}')
print(f'{"S2 (Attack)":<12} | {single_s2:<12.4f} | {mcc_s2:<14.4f} | {mcc_s2-single_s2:+.4f}')
print(f'{"S3 (Integ.)":<12} | {single_s3:<12.4f} | {mcc_s3:<14.4f} | {mcc_s3-single_s3:+.4f}')
print(f'{"S4 (Robust)":<12} | {single_s4:<12.4f} | {mcc_s4:<14.4f} | {mcc_s4-single_s4:+.4f}')
print('='*50)

## 6. Visualisasi

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

scenarios = ['S1\n(Clean)', 'S2\n(Attack)', 'S3\n(Integrity)', 'S4\n(Robust)']
x = np.arange(len(scenarios))
width = 0.35

single_mcc = [single_s1, single_s2, single_s3, single_s4]
stack_mcc = [mcc_s1, mcc_s2, mcc_s3, mcc_s4]

bars1 = ax.bar(x - width/2, single_mcc, width, label='Single XGBoost', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, stack_mcc, width, label='Stacking Ensemble', color='darkorange', alpha=0.8)

ax.set_ylabel('MCC', fontsize=12)
ax.set_title('S1-S4 Comparison: Single XGBoost vs Stacking Ensemble', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.1)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'stacking_s1_s4_comparison.png'), bbox_inches='tight')
plt.show()
print('Saved: stacking_s1_s4_comparison.png')

## 7. Save Results

In [ ]:
output = {
    # Stacking S1-S4
    'stacking_s1': {'mcc': mcc_s1, 'f1': f1_s1},
    'stacking_s2': {'mcc': mcc_s2, 'f1': f1_s2},
    'stacking_s3': {'mcc': mcc_s3, 'f1': f1_s3},
    'stacking_s4': {'mcc': mcc_s4, 'f1': f1_s4},
    
    # Single model S1-S4 (from notebook 06)
    'single_s1': single_s1, 'single_s2': single_s2,
    'single_s3': single_s3, 'single_s4': single_s4,
    
    # Robust models
    'robust_models': robust_models,
    'robust_meta': robust_meta,
    'robust_scaler': robust_scaler,
    
    # Config
    'epsilon_train': EPSILON_TRAIN,
    'train_time': train_time
}

with open(os.path.join(DATA_DIR, 'stacking_robust_18.pkl'), 'wb') as f:
    pickle.dump(output, f)

print('Saved: stacking_robust_18.pkl')
print('\nNotebook 18 selesai. Lanjut ke 19 (Robustness Ablation) atau 20 (Final Evaluation).')